## Langchain Expression Language Basics

-  LangChain Expression Language is that any two runnables can be "chained" together into sequences. 
- The output of the previous runnable's .invoke() call is passed as input to the next runnable.
- This can be done using the pipe operator (|), or the more explicit .pipe() method, which does the same thing.

- Type of LCEL Chains
    - SequentialChain
    - Parallel Chain
    - Router Chain
    - Chain Runnables
    - Custom Chain (Runnable Sequence)

In [1]:
from dotenv import load_dotenv

load_dotenv('./../.env')

True

### Sequential LCEL Chain

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate
                                        )

base_url = "http://localhost:11434"
model = 'qwen3'

llm = ChatOllama(base_url=base_url, model=model)
llm

ChatOllama(model='qwen3', base_url='http://localhost:11434')

In [3]:
system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')


messages = [system, question]
template = ChatPromptTemplate(messages)

question = template.invoke({'school': 'primary', 'topics': 'solar system', 'points': 5})

response = llm.invoke(question)
print(response.content)    

1. The Sun is the center, holding the solar system together with gravity.  
2. Eight planets orbit the Sun: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.  
3. Pluto is a dwarf planet, not a full planet, located beyond Neptune.  
4. The asteroid belt lies between Mars and Jupiter, with rocky objects.  
5. The Kuiper Belt and Oort Cloud are regions of icy bodies, including comets.


In [4]:
system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')


messages = [system, question]
template = ChatPromptTemplate(messages)

chain = template | llm


In [5]:
response = chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 5})
print(response.content)

1. The Sun is the center, holding the solar system together with gravity.  
2. Eight planets orbit the Sun: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.  
3. The asteroid belt lies between Mars and Jupiter, with rocky objects.  
4. Comets originate from the Kuiper Belt (near Neptune) and the distant Oort Cloud.  
5. The solar system includes moons, dwarf planets, and other celestial bodies.


In [6]:
response = chain.invoke({'school': 'phd', 'topics': 'solar system', 'points': 5})
print(response.content)

1. The Sun is the central star, providing gravity and energy.  
2. Eight planets orbit the Sun: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.  
3. Smaller bodies like asteroids, comets, and dwarf planets (e.g., Pluto) exist in the solar system.  
4. The asteroid belt and Kuiper Belt are regions with numerous small objects.  
5. The heliosphere marks the boundary of the Sun's influence in space.


In [7]:
response

AIMessage(content="1. The Sun is the central star, providing gravity and energy.  \n2. Eight planets orbit the Sun: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.  \n3. Smaller bodies like asteroids, comets, and dwarf planets (e.g., Pluto) exist in the solar system.  \n4. The asteroid belt and Kuiper Belt are regions with numerous small objects.  \n5. The heliosphere marks the boundary of the Sun's influence in space.", additional_kwargs={}, response_metadata={'model': 'qwen3', 'created_at': '2026-03-20T09:51:53.015525Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3818223400, 'load_duration': 92150300, 'prompt_eval_count': 37, 'prompt_eval_duration': 320515600, 'eval_count': 348, 'eval_duration': 3248615300, 'logprobs': None, 'model_name': 'qwen3', 'model_provider': 'ollama'}, id='lc_run--019d0aa8-83cb-7730-9536-d05c5fddb9f5-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 348, 'total_tokens': 385})

In [9]:
from langchain_core.output_parsers import StrOutputParser

In [10]:
chain = template | llm | StrOutputParser()
response = chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 5})
print(response)

1. The Sun is the central star.  
2. Eight planets orbit it: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune.  
3. Asteroid belt lies between Mars and Jupiter.  
4. Kuiper Belt and Oort Cloud are comet regions.  
5. All objects are bound by gravity.


In [10]:
response

'1. The Sun is the center, holding the solar system together with gravity.  \n2. Eight planets orbit the Sun: Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune.  \n3. The asteroid belt lies between Mars and Jupiter, with rocky objects.  \n4. Moons orbit planets, like Earth’s Moon and Jupiter’s many moons.  \n5. Distant regions include the Kuiper Belt and Oort Cloud, home to icy bodies.'

### Chaining Runnables (Chain Multiple Runnables)

- We can even combine this chain with more runnables to create another chain.
- Let's see how easy our generated output is?

In [11]:
chain

ChatPromptTemplate(input_variables=['points', 'school', 'topics'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['school'], input_types={}, partial_variables={}, template='You are {school} teacher. You answer in short sentences.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['points', 'topics'], input_types={}, partial_variables={}, template='tell me about the {topics} in {points} points'), additional_kwargs={})])
| ChatOllama(model='qwen3', base_url='http://localhost:11434')
| StrOutputParser()

In [12]:
analysis_prompt = ChatPromptTemplate.from_template('''analyze the following text: {response}
                                                   You need tell me that how difficult it is to understand.
                                                   Answer in one sentence only.
                                                   ''')

fact_check_chain = analysis_prompt | llm | StrOutputParser()
output = fact_check_chain.invoke({'response': response})
print(output)

The text is very easy to understand as it presents basic, straightforward astronomical facts without complex terminology or abstract concepts.


In [13]:
composed_chain = {"response": chain} | analysis_prompt | llm | StrOutputParser()

output = composed_chain.invoke({'school': 'phd', 'topics': 'solar system', 'points': 5})
print(output)

The text is moderately difficult to understand due to some technical terms like "Kuiper Belt," "Oort Cloud," and "heliosphere," but most concepts are accessible with basic science knowledge.


### Parallel LCEL Chain
- Parallel chains are used to run multiple runnables in parallel.
- The final return value is a dict with the results of each value under its appropriate key.

In [14]:
system = SystemMessagePromptTemplate.from_template('You are {school} teacher. You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')


messages = [system, question]
template = ChatPromptTemplate(messages)
fact_chain = template | llm | StrOutputParser()

output = fact_chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 2})
print(output)

1. The solar system is centered around the Sun, with planets, moons, and other objects orbiting it.  
2. It includes eight major planets, dwarf planets, asteroids, comets, and dust.


In [15]:
question = HumanMessagePromptTemplate.from_template('write a poem on {topics} in {sentences} lines')


messages = [system, question]
template = ChatPromptTemplate(messages)
poem_chain = template | llm | StrOutputParser()

output = poem_chain.invoke({'school': 'primary', 'topics': 'solar system', 'sentences': 2})
print(output)

The sun shines bright, a golden core,  
Planets dance, moons glow in their orbit's lore.


In [17]:
from langchain_core.runnables import RunnableParallel

In [18]:
chain = RunnableParallel(fact = fact_chain, poem = poem_chain)

In [19]:
output = chain.invoke({'school': 'primary', 'topics': 'solar system', 'points': 2, 'sentences': 2})
print(output['fact'])
print('\n\n')
print(output['poem'])

1. The solar system revolves around the Sun, with eight planets orbiting it.  
2. It includes moons, asteroids, comets, and other celestial objects.



The sun, a golden heart, spins bright,  
Planets dance in silent flight.


### Chain Router
- The router chain is used to route the output of a previous runnable to the next runnable based on the output of the previous runnable.

In [20]:
prompt = """Given the user review below, classify it as either being about `Positive` or `Negative`.
            Do not respond with more than one word.

            Review: {review}
            Classification:"""

template = ChatPromptTemplate.from_template(prompt)

chain = template | llm | StrOutputParser()

review = "Thank you so much for providing such a great platform for learning. I am really happy with the service."
# review = "I am not happy with the service. It is not good."
chain.invoke({'review': review})

'Positive'

In [21]:
positive_prompt = """
                You are expert in writing reply for positive reviews.
                You need to encourage the user to share their experience on social media.
                Review: {review}
                Answer:"""

positive_template = ChatPromptTemplate.from_template(positive_prompt)
positive_chain = positive_template | llm | StrOutputParser()

In [22]:
negative_prompt = """
                You are expert in writing reply for negative reviews.
                You need first to apologize for the inconvenience caused to the user.
                You need to encourage the user to share their concern on following Email:'udemy@kgptalkie.com'.
                Review: {review}
                Answer:"""


negative_template = ChatPromptTemplate.from_template(negative_prompt)
negative_chain = negative_template | llm | StrOutputParser()

In [23]:
def route(info):
    if 'positive' in info['sentiment'].lower():
        return positive_chain
    else:
        return negative_chain

In [23]:
# rout({'sentiment': 'negetive'})

In [24]:
from langchain_core.runnables import RunnableLambda

In [25]:
full_chain = {"sentiment": chain, 'review': lambda x: x['review']} | RunnableLambda(route)

In [26]:
full_chain

{
  sentiment: ChatPromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['review'], input_types={}, partial_variables={}, template='Given the user review below, classify it as either being about `Positive` or `Negative`.\n            Do not respond with more than one word.\n\n            Review: {review}\n            Classification:'), additional_kwargs={})])
             | ChatOllama(model='qwen3', base_url='http://localhost:11434')
             | StrOutputParser(),
  review: RunnableLambda(lambda x: x['review'])
}
| RunnableLambda(route)

In [27]:
# review = "Thank you so much for providing such a great plateform for learning. I am really happy with the service."
review = "I am not happy with the service. It is not good."

output = full_chain.invoke({'review': review})
print(output)

**Answer:**  
Dear [User],  
We sincerely apologize for the inconvenience and disappointment caused by your experience. We truly value your feedback and are committed to improving our services. To better address your concerns, we kindly encourage you to share your detailed feedback at **udemy@kgptalkie.com**. Your input is invaluable, and we’ll do our best to resolve the issue promptly. Thank you for bringing this to our attention.  
Best regards,  
[Your Name/Team]


### Make Custom Chain Runnables with RunnablePassthrough and RunnableLambda
- This is useful for formatting or when you need functionality not provided by other LangChain components, and custom functions used as Runnables are called RunnableLambdas.



In [28]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [29]:
def char_counts(text):
    return len(text)

def word_counts(text):
    return len(text.split())

prompt = ChatPromptTemplate.from_template("Explain these inputs in 5 sentences: {input1} and {input2}")

In [30]:
prompt

ChatPromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input1', 'input2'], input_types={}, partial_variables={}, template='Explain these inputs in 5 sentences: {input1} and {input2}'), additional_kwargs={})])

In [31]:
chain = prompt | llm | StrOutputParser()

output = chain.invoke({'input1': 'Earth is planet', 'input2': 'Sun is star'})

print(output)

1. Earth is a planet, which is a celestial body that orbits a star, in this case, the Sun.  
2. Planets like Earth are distinct from stars because they do not produce their own light through nuclear fusion.  
3. The Sun is a star, a massive, luminous sphere of plasma that generates energy through nuclear fusion in its core.  
4. Stars like the Sun emit light and heat, which are essential for sustaining life on planets such as Earth.  
5. The relationship between Earth and the Sun defines the solar system, with Earth's orbit around the Sun enabling conditions for life.


In [33]:
chain = prompt | llm | StrOutputParser() | {'char_counts': RunnableLambda(char_counts), 
                                            'word_counts': RunnableLambda(word_counts), 
                                            'output': RunnablePassthrough()}

output = chain.invoke({'input1': 'Earth is planet', 'input2': 'Sun is star'})

print(output)

{'char_counts': 703, 'word_counts': 120, 'output': "1. Earth is a planet, one of the eight celestial bodies in our solar system, characterized by its solid surface, atmosphere, and ability to support life.  \n2. The Sun is a star, a massive, luminous sphere of plasma held together by gravity, emitting light and heat through nuclear fusion.  \n3. Earth orbits the Sun in an elliptical path, completing one revolution every 365 days, which defines a year.  \n4. While Earth is a rocky planet with a relatively small size, the Sun is a massive star, about 109 times wider than Earth.  \n5. Together, Earth and the Sun form a dynamic system where the Sun's energy sustains life on Earth, highlighting their distinct yet interconnected roles in the solar system."}


### Custom Chain using `@chain` decorator

In [35]:
from langchain_core.runnables import chain

In [36]:
@chain
def custom_chain(params):
    return {
        'fact': fact_chain.invoke(params),
        'poem': poem_chain.invoke(params),
    }


params = {'school': 'primary', 'topics': 'solar system', 'points': 2, 'sentences': 2}
output = custom_chain.invoke(params)
print(output['fact'])
print('\n\n')
print(output['poem'])

1. The solar system revolves around the Sun, with eight planets orbiting it.  
2. It includes moons, asteroids, comets, and other celestial bodies, spanning vast distances.



The sun, a golden heart, spins bright,  
Planets dance in endless light.
